In [ ]:
# ============================================================
# セル 1: GPU 確認
# ============================================================
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    raise SystemExit('GPU が見つかりません。ランタイムを GPU に変更してください。')

In [ ]:
# ============================================================
# セル 2: Rust インストール
# ============================================================
import subprocess
import os
import glob

subprocess.run(
    'curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable',
    shell=True,
)

os.environ['PATH'] = f"/root/.cargo/bin:{os.environ['PATH']}"

cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

for cmd in [['rustc', '--version'], ['cargo', '--version']]:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.strip() if r.returncode == 0 else f'{cmd[0]} が見つかりません')

In [ ]:
# ============================================================
# セル 3: リポジトリ clone
# Colab シークレットに GITHUB_TOKEN（Fine-grained PAT）を設定
# 権限: Contents (read/write)
# ============================================================
import subprocess
import os
from google.colab import userdata

REPO_DIR = '/content/puyopuyo-ai'
BRANCH = 'develop'

token = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{token}@github.com/hfappmaker/puyopuyo-ai.git'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('リポジトリ既存 → git pull')
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    subprocess.run(['git', 'pull'], cwd=REPO_DIR)
else:
    print(f'clone 中（ブランチ: {BRANCH}）...')
    result = subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR])
    if result.returncode != 0:
        raise RuntimeError('clone 失敗。GITHUB_TOKEN を確認してください。')
    print('clone 完了')

subprocess.run(['git', 'config', 'user.email', 'colab-training@example.com'], cwd=REPO_DIR)
subprocess.run(['git', 'config', 'user.name', 'Colab Training'], cwd=REPO_DIR)
subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_DIR)

In [ ]:
# ============================================================
# セル 4: AlphaZero ループ実行
# scripts/alphazero-loop-push.sh が self-play → train を無限ループで実行し、
# 各イテレーション後に自動で git commit & push します。
# 環境変数でパラメータを変更できます。
# ============================================================
import subprocess
import os
import glob

REPO_DIR = '/content/puyopuyo-ai'
BRANCH = 'develop'

os.environ['PATH'] = f"/root/.cargo/bin:{os.environ['PATH']}"
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

# --- ゲーム設定 ---
os.environ['BOARD_COLS'] = '3'
os.environ['BOARD_ROWS'] = '8'
os.environ['NUM_COLORS'] = '3'

# --- モデル設定 ---
os.environ['RESIDUAL_CHANNELS'] = '64'
os.environ['NUM_BLOCKS'] = '6'
os.environ['POLICY_CONV_CHANNELS'] = '2'
os.environ['VALUE_CONV_CHANNELS'] = '1'
os.environ['VALUE_HIDDEN'] = '64'
os.environ['FILM_HIDDEN'] = '128'

# --- 学習パラメータ ---
os.environ['GAMES'] = '300'
os.environ['SIMS_BASE'] = '64'
os.environ['SIMS_STEP'] = '0'
os.environ['SIMS_MAX'] = '64'
os.environ['C_PUCT_INIT'] = '1.5'
os.environ['C_PUCT_BASE'] = '19652.0'
os.environ['M'] = '16'
os.environ['C_VISIT'] = '50.0'
os.environ['GAMMA'] = '0.95'
os.environ['REPLAY_WINDOW'] = '30'
os.environ['MIN_CHAIN'] = '0'
os.environ['THREADS'] = '128'
os.environ['INFER_BATCH_SIZE'] = '128'
os.environ['NUM_LEAVES'] = '1'             # MCTS virtual loss バッチ（同時探索リーフ数）
os.environ['TRAIN_BATCH_SIZE'] = '512'
os.environ['ACCUM_STEPS'] = '4'

process = subprocess.Popen(
    ['bash', 'scripts/alphazero-loop-push.sh', BRANCH],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ,
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
if process.returncode != 0:
    print(f'alphazero-loop-push.sh 失敗 (returncode={process.returncode})')